In [1]:
import numpy as np
import statistics
import pandas as pd
from PIL import Image as PILImage
from pathlib import Path
import random
from dataclasses import dataclass, field

In [2]:
@dataclass
class GlobalState:
    F_1: pd.DataFrame = field(default_factory=lambda: pd.DataFrame(columns=["p","v","t"]))
    V_1: dict = field(default_factory=dict)
    L: int = 0
    last: int = 1
    ell: int = 0
    input_folder: str = r"G:\MorphologyScaleSpace\images"
    output_folder: str = r"G:\MorphologyScaleSpace\results"

    def reset(self):
        self.F_1 = pd.DataFrame(columns=["p","v","t"])
        self.V_1 = {}
        self.L = 0
        self.last = 1
        self.ell = 0

state = GlobalState()

In [3]:
class Image:
    def __init__(self, data):
        # Store the data as a NumPy array for fast math operations
        self._data = np.array(data)

    @property
    def height(self):
        # The number of rows
        return self._data.shape[0]

    @property
    def width(self):
        # The number of columns
        return self._data.shape[1]

    def __getitem__(self, indices):
        # Allows reading pixels using image[i, j]
        i, j = indices
        return self._data[i, j]

    def __setitem__(self, indices, value):
        # Allows modifying pixels using image[i, j] = value
        i, j = indices
        self._data[i, j] = value

    def as_array(self):
        # Helper to return the raw matrix when you need to do full-image math
        return self._data

    def __sub__(self, other: Image):
        arr_self = self.as_array()
        arr_other = other.as_array()

        return Image(arr_self - arr_other)

    def __add__(self, other: Image):
        arr_self = self.as_array()
        arr_other = other.as_array()

        return Image(arr_self + arr_other)

    def save_bmp(self, filepath):
        raw_array = self.as_array()
        visual_array = np.clip(raw_array, 0, 255).astype(np.uint8)
        PILImage.fromarray(visual_array).save(filepath)

    def absolute_log_difference(self, other: Image) -> Image:
        arr_self = self.as_array()
        arr_other = other.as_array()

        abs_error = np.abs(arr_self - arr_other)

        c = 255 / np.log1p(255)
        log_error = c * np.log1p(abs_error)

        discrete_log_error = np.round(log_error)

        return Image(discrete_log_error)


In [4]:
def expand(u_t):
    target_h = 2 * u_t.height
    target_w = 2 * u_t.width

    # Initialize the target approximation image (Omega_{t-1})
    u_t_minus_one = Image(np.zeros((target_h, target_w), dtype=int))

    # Boundary Condition: Edge Replication (Zero-order hold)
    def get_p(r, c):
        r_clamp = max(0, min(r, u_t.height - 1))
        c_clamp = max(0, min(c, u_t.width - 1))
        return u_t[r_clamp, c_clamp]

    # Iterate over the TARGET grid (i, j)
    for i in range(target_h):
        for j in range(target_w):

            # Map to SOURCE grid (r, c)
            r = i // 2
            c = j // 2

            # 1. Base Pixel (Direct Copy)
            if i % 2 == 0 and j % 2 == 0:
                u_t_minus_one[i, j] = get_p(r, c)

            # 2. Horizontal Edge Interpolation
            elif i % 2 == 0 and j % 2 == 1:
                median_list = [
                    get_p(r-1, c),   get_p(r-1, c+1),
                    get_p(r, c),     get_p(r, c),     get_p(r, c),
                    get_p(r, c+1),   get_p(r, c+1),   get_p(r, c+1),
                    get_p(r+1, c),   get_p(r+1, c+1)
                ]
                u_t_minus_one[i, j] = statistics.median_low(median_list)

            # 3. Vertical Edge Interpolation
            elif i % 2 == 1 and j % 2 == 0:
                median_list = [
                    get_p(r, c-1),   get_p(r+1, c-1),
                    get_p(r, c),     get_p(r, c),     get_p(r, c),
                    get_p(r+1, c),   get_p(r+1, c),   get_p(r+1, c),
                    get_p(r, c+1),   get_p(r+1, c+1)
                ]
                u_t_minus_one[i, j] = statistics.median_low(median_list)

            # 4. Diagonal Center Interpolation
            else:
                median_list = [
                    get_p(r, c),     get_p(r+1, c),
                    get_p(r, c+1),   get_p(r+1, c+1)
                ]
                u_t_minus_one[i, j] = statistics.median_low(median_list)

    return u_t_minus_one

def decimate(u_t):
    # Base case: The image cannot be decimated further
    if u_t.width == 1 and u_t.height == 1:
        return u_t

    target_h = (u_t.height + 1) // 2
    target_w = (u_t.width + 1) // 2

    # Initialize the target decimated image (Omega_{t+1})
    u_t_plus_one = Image(np.zeros((target_h, target_w), dtype=int))

    # Iterate strictly over the smaller target grid (r, c)
    for r in range(target_h):
        for c in range(target_w):

            # Map back to the source grid (i, j)
            i = 2 * r
            j = 2 * c

            # Direct sub-sampling
            u_t_plus_one[r, c] = u_t[i, j]

    return u_t_plus_one

In [5]:
def compute_residual_t(u_t, est_u_t, t):

    # 1. Vectorized Subtraction
    r_t_array = u_t - est_u_t

    # Temporary list to hold the new tuples for this level
    new_tuples = []

    # 2. Iteration and Non-Expansive Filtering
    for i in range(u_t.height):
        for j in range(u_t.width):
            # Skip the sub-sampled base pixels
            if i % 2 != 0 or j % 2 != 0:
                p = (i, j)
                v = r_t_array[i, j]

                new_tuples.append({"p": p, "v": v, "t": t})

    # 3. Batch Update the Feature Set
    if new_tuples:
        new_df = pd.DataFrame(new_tuples)
        # Using concat updates the global DataFrame reference
        state.F_1 = pd.concat([state.F_1, new_df], ignore_index=True)

In [6]:
def compute_a_t(t):
    a_t = state.F_1[state.F_1['t'] == t]['v'].unique()
    state.V_1[t] = a_t

In [7]:
def feature_generator(image):
    u_t = image
    t = 1

    while True:
        u_t_plus_one = decimate(u_t)

        # Base Case condition utilizing dimension check
        if u_t_plus_one.width == u_t.width and u_t_plus_one.height == u_t.height:
            new_tuples = []
            for i in range(u_t.height):
                for j in range(u_t.width):
                    new_tuples.append({"p": (i, j), "v": u_t[i, j], "t": t})

            if new_tuples:
                state.F_1 = pd.concat([state.F_1, pd.DataFrame(new_tuples)], ignore_index=True)

            compute_a_t(t)
            state.last = t
            break

        est_u_t = expand(u_t_plus_one)

        compute_residual_t(u_t, est_u_t, t)
        compute_a_t(t)

        t += 1
        u_t = u_t_plus_one

    # L calculation evaluating the Restricted Zeros assumption
    sigma_A_t = 0
    for i in range(1, state.last + 1):
        sigma_A_t += len(state.V_1[i])

    zeros_count = len(state.F_1[state.F_1["v"] == 0])
    state.L = (zeros_count - 1) + (sigma_A_t - 1) - 1

In [8]:
def compute_level_dimensions(h_0, w_0, target_t):
    h, w = h_0, w_0
    for _ in range(1, target_t):
        h = (h + 1) // 2
        w = (w + 1) // 2
    return h, w

def reconstruct_residual_t(height_t, width_t, t):
    # 1. Block Memory Allocation: Instantiate Omega_t
    r_t_array = np.zeros((height_t, width_t), dtype=int)

    # 2. Isolate the subset P_t (the surviving features for level t)
    f_t = state.F_1.loc[state.F_1['t'] == t]

    # 3. Sparse Matrix Population
    for p, v in zip(f_t["p"], f_t["v"]):
        i, j = p
        r_t_array[i, j] = v

    return Image(r_t_array)

def export_pyramid_gradient(image_name, original_height, original_width):

    results_dir = Path("results")
    results_dir.mkdir(exist_ok=True)

    # Iterate through every level up to the final base pixel
    for t in range(1, state.last + 1):
        h, w = compute_level_dimensions(original_height, original_width, t)

        # 1. Allocate an RGBA matrix (4 channels)
        rgba_array = np.zeros((h, w, 4), dtype=np.uint8)

        # 2. Handle the final level (base pixel) vs. residual levels
        if t == state.last:
            base_df = state.F_1[state.F_1["t"] == state.last]
            if not base_df.empty:
                base_v = int(base_df["v"].values[0])
                base_v_clamped = max(0, min(255, base_v))
                # Base pixel is fully opaque grayscale
                rgba_array[0, 0] = [base_v_clamped, base_v_clamped, base_v_clamped, 255]
        else:
            # Reconstruct the sparse residual matrix
            r_t_image = reconstruct_residual_t(h, w, t)
            r_t_array = r_t_image.as_array()

            # 3. Compute absolute magnitude of the prediction error
            # np.abs safely handles the negative values in the signed integer array
            abs_r_t = np.abs(r_t_array)
            abs_r_t_clamped = np.clip(abs_r_t, 0, 255).astype(np.uint8)

            # Map the absolute magnitude equally to R, G, and B for grayscale
            rgba_array[..., 0] = abs_r_t_clamped
            rgba_array[..., 1] = abs_r_t_clamped
            rgba_array[..., 2] = abs_r_t_clamped

            # Make all calculated residuals strictly opaque by default
            rgba_array[..., 3] = 255

            # Mark strictly decimated coordinates as fully transparent (Alpha = 0)
            rgba_array[0::2, 0::2, 3] = 0

        # 4. Save the isolated image
        output_path = results_dir / f"{image_name}_level_{t}.png"
        PILImage.fromarray(rgba_array, mode="RGBA").save(output_path)

    print(f"Exported {state.last} absolute magnitude level images for {image_name}.")

In [9]:
def uncommitted_uniform_quantization(a_t: list[int], t: int, t_arr: np.ndarray, v_arr: np.ndarray, active_mask: np.ndarray):
    min_diff = float('inf')
    merge_idx = 0

    for i in range(len(a_t) - 1):
        diff = a_t[i + 1] - a_t[i]
        if diff < min_diff:
            min_diff = diff
            merge_idx = i

    q1 = a_t[merge_idx]
    q2 = a_t[merge_idx + 1]
    s = q1 + q2
    # sgn(s) * ((|s| + 1) / 2)
    q_new = int(np.sign(s) * ((abs(s) + 1) // 2))

    mask_1 = (t_arr == t) & (v_arr == q1) & active_mask
    mask_2 = (t_arr == t) & (v_arr == q2) & active_mask

    v_arr[mask_1 | mask_2] = q_new


def committed_ward_clustering(a_t: list[int], t: int, t_arr: np.ndarray, v_arr: np.ndarray, active_mask: np.ndarray):
    best_q1, best_q2, best_q_new = None, None, None
    min_mse_increase = float('inf')

    for i in range(len(a_t) - 1):
        q1 = a_t[i]
        q2 = a_t[i + 1]

        mask_1 = (t_arr == t) & (v_arr == q1) & active_mask
        mask_2 = (t_arr == t) & (v_arr == q2) & active_mask

        count_q1 = np.sum(mask_1)
        count_q2 = np.sum(mask_2)
        q_new = q1 if count_q1 >= count_q2 else q2

        mse_increase = count_q1 * ((q1 - q_new) ** 2) + count_q2 * ((q2 - q_new) ** 2)

        if mse_increase < min_mse_increase:
            min_mse_increase = mse_increase
            best_q1 = q1
            best_q2 = q2
            best_q_new = q_new

    mask_1 = (t_arr == t) & (v_arr == best_q1) & active_mask
    mask_2 = (t_arr == t) & (v_arr == best_q2) & active_mask

    v_arr[mask_1 | mask_2] = best_q_new

In [72]:
def step_function(t: int, operation: str, t_arr: np.ndarray, v_arr: np.ndarray, active_mask: np.ndarray):
    match operation:
        case "Sparsification":
            drop_mask = (t_arr == t) & (v_arr == 0) & active_mask
            num_dropped = np.sum(drop_mask)

            if num_dropped > 0:
                active_mask[drop_mask] = False
                state.ell += num_dropped

        case "Quantization":
            active_v = v_arr[(t_arr == t) & active_mask & (v_arr != 0)]
            a_t = sorted(np.unique(active_v))
            if len(a_t) <= 1:
                return

            # Options:
            # uncommitted_uniform_quantization(a_t, t, t_arr, v_arr, active_mask)
            committed_ward_clustering(a_t, t, t_arr, v_arr, active_mask)

            state.ell += 1

In [73]:
def compression(t_arr: np.ndarray, v_arr: np.ndarray, active_mask: np.ndarray):
    if state.L == state.ell:
        return False

    random_t = random.randint(1, state.last - 1)
    random_op = random.choice(["Sparsification", "Quantization"])
    step_function(random_t, random_op, t_arr, v_arr, active_mask)

    return True

In [12]:
def generate_u_last() -> Image:
    f_last = state.F_1.loc[state.F_1['t'] == state.last, ['p', 'v']]

    max_i = max(p[0] for p in f_last['p'])
    max_j = max(p[1] for p in f_last['p'])

    u_last_array = np.zeros((max_i + 1, max_j + 1), dtype=int)

    for p, v in zip(f_last["p"], f_last["v"]):
        i, j = p
        u_last_array[i, j] = v

    return Image(u_last_array)

In [13]:
def generate_r_t(t: int, width_t: int, height_t: int) -> Image:
    r_t_array = np.zeros((height_t, width_t), dtype=int)
    f_t = state.F_1.loc[state.F_1['t'] == t]

    for p, v in zip(f_t["p"], f_t["v"]):
        i, j = p
        r_t_array[i, j] = v

    return Image(r_t_array)

In [14]:
def generate_u_one() -> Image:
    u_current = generate_u_last()

    for t in range(state.last - 1, 0, -1):
      est_u = expand(u_current)
      r_t = generate_r_t(t, est_u.width, est_u.height)
      u_current = est_u + r_t

    return u_current

In [74]:
def process_image_folder(folder_path):
    # Create a Path object for robust directory handling
    directory = Path(folder_path)

    # Iterate specifically over all BMP files in the folder
    for file_path in directory.glob("*.bmp"):
        print(f"--- Starting Pipeline for: {file_path.name} ---")

        # 1. Purge previous image data
        state.reset()

        raw_img = PILImage.open(str(file_path))
        pixel_matrix = np.array(raw_img, dtype=int)
        f = Image(pixel_matrix)

        # 2. Execute pipeline (cast to string for PIL compatibility)
        feature_generator(f)

        t_arr = state.F_1["t"].values.astype(int)
        v_arr = state.F_1["v"].values.astype(int)
        active_mask = np.ones(len(state.F_1), dtype=bool)
        #export_pyramid_gradient(file_path.stem, original_h, original_w)

        # 2. Step function
        for i in range(100):
            compression(t_arr, v_arr, active_mask)

        state.F_1["v"] = v_arr
        state.F_1 = state.F_1[active_mask].reset_index(drop=True)

        # 3. Reconstruction
        final_u_1 = generate_u_one()

        path = Path(state.output_folder) / f"{file_path.name}"
        final_u_1.save_bmp(path)

        diff_image = f.absolute_log_difference(final_u_1)
        diff_image.save_bmp(Path(state.output_folder) / f"Diff {file_path.name}")

        print(f"    Compressed: {state.ell / state.L}%")

        print(f"--- Ending Pipeline for: {file_path.name} ---")

In [50]:
state.F_1

,p,v,t
0,"(0, 1)",0,1
1,"(0, 3)",-2,1
2,"(0, 5)",-6,1
3,"(0, 7)",-4,1
4,"(0, 9)",-2,1
...,...,...,...
262139,"(3, 3)",20,8
262140,"(0, 1)",-40,9
262141,"(1, 0)",-40,9
262142,"(1, 1)",-40,9


In [75]:
process_image_folder(state.input_folder)

--- Starting Pipeline for: lena.bmp ---
    Compressed: 0.9672063229013959%
--- Ending Pipeline for: lena.bmp ---
